### Import Libraries and Reproducibility

In [ ]:
import sys, os, re, json, pandas as pd, numpy as np
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm
import transformers

SEED = 22
transformers.set_seed(SEED)
rng = np.random.default_rng(SEED)

### Configuration and Instructions

In [ ]:
TRAIN_PATH = "train_w4.csv"
TEST_PATH  = "test_w4.csv"
LABEL      = "SNAP"
OUTPUT_DIR = "./results_wave4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

# Feature sets
# SNAP_wave1, SNAP_wave2 and SNAP_wave3 are LONGITUDINAL predictors: self-reported Wave 1, 2 and 3 SNAP
# participation carried forward into the Wave 4 feature set.

DEMO_FEATURES     = [
    "marital.status", "sex", "hispanic.origin", "race",
    "education", "citizenship", "employment", "state", "age", "income"
]
LONGITUDINAL_FEATURES = ["SNAP_wave1"] + ["SNAP_wave2"] + ["SNAP_wave3"]
FULL_FEATURES     = DEMO_FEATURES + LONGITUDINAL_FEATURES          # Wave 4 full model

COL_MAP = {
    "marital.status":  "Marital Status",
    "sex":             "Sex",
    "hispanic.origin": "Hispanic Origin",
    "race":            "Race",
    "education":       "Educational Attainment",
    "citizenship":     "Citizenship",
    "employment":      "Employment Status",
    "state":           "State of Residence",
    "age":             "Age",
    "income":          "Income (Wave 3)",
    "SNAP_wave1":      "SNAP Coverage in Wave 1 (Prior Wave)",
    "SNAP_wave2":      "SNAP Coverage in Wave 2 (Prior Wave)",
    "SNAP_wave3":      "SNAP Coverage in Wave 3 (Most Recent Prior Wave)",
    "SNAP":            "Coverage for SNAP in Wave 4",
}

#  Instructions
INSTRUCTION_STRUCTURED = (
    "You are an expert text classifier. "
    "The Supplemental Nutrition Assistance Program (SNAP) provides food-purchasing "
    "assistance to low-income individuals and families in the U.S. "
    "You will be given a respondent's demographic information and income from "
    "Wave 4 of the 2014 Survey of Income and Program Participation (SIPP), "
    "along with whether they had SNAP coverage in Wave 1 (first prior survey wave), Wave 2 (second prior survey wave) and in Wave 3 (the most recent prior survey wave)."
    "This Wave 1, 2 and 3 SNAP status captures prior "
    "program participation and benefit receipt for that individual. "
    "Based on all this information, classify if the person has SNAP coverage in Wave 4. "
    "Return only one label: 'Yes' or 'No', without any other text.\n"
)

INSTRUCTION_NARRATIVE = (
    "You are an expert text classifier. "
    "The Supplemental Nutrition Assistance Program (SNAP) provides food assistance "
    "to low-income households in the U.S. "
    "You will read a short profile of a survey respondent from the 2014 Survey of "
    "Income and Program Participation (SIPP), Wave 4. "
    "The profile includes whether the respondent received SNAP in Wave 3 (the "
    "immediately preceding survey wave), in Wave 2 (the wave prior to wave 3) and in Wave 1 (the wave prior to wave 2). "
    "Based on the full profile, decide whether this person has SNAP coverage in Wave 4. "
    "Return only one label: 'Yes' or 'No', without any other text.\n"
)

### Data Loading

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train : {len(df_train):,} rows | SNAP distribution: {dict(Counter(df_train[LABEL]))}")
print(f"Test  : {len(df_test):,} rows  | SNAP distribution: {dict(Counter(df_test[LABEL]))}")

# check that the longitudinal predictors are present
assert "SNAP_wave1" in df_train.columns, "SNAP_wave1 column missing from train_w4.csv"
assert "SNAP_wave1" in df_test.columns,  "SNAP_wave1 column missing from test_w4.csv"
print("\nSNAP_wave1 confirmed in both splits.")
print(f"Train SNAP_wave1 dist : {dict(Counter(df_train['SNAP_wave1']))}")
print(f"Test  SNAP_wave1 dist : {dict(Counter(df_test['SNAP_wave1']))}")

assert "SNAP_wave2" in df_train.columns, "SNAP_wave2 column missing from train_w4.csv"
assert "SNAP_wave2" in df_test.columns,  "SNAP_wave2 column missing from test_w4.csv"
print("\nSNAP_wave2 confirmed in both splits.")
print(f"Train SNAP_wave2 dist : {dict(Counter(df_train['SNAP_wave2']))}")
print(f"Test  SNAP_wave2 dist : {dict(Counter(df_test['SNAP_wave2']))}")

assert "SNAP_wave3" in df_train.columns, "SNAP_wave3 column missing from train_w4.csv"
assert "SNAP_wave3" in df_test.columns,  "SNAP_wave3 column missing from test_w4.csv"
print("\nSNAP_wave3 confirmed in both splits.")
print(f"Train SNAP_wave3 dist : {dict(Counter(df_train['SNAP_wave3']))}")
print(f"Test  SNAP_wave3 dist : {dict(Counter(df_test['SNAP_wave3']))}")

### Oversampling

In [ ]:
def oversample_minority(df, label_col, seed=SEED):
    majority = df[df[label_col] == "No"]
    minority = df[df[label_col] == "Yes"]
    minority_upsampled = minority.sample(
        n=len(majority), replace=True, random_state=seed
    )
    return pd.concat([majority, minority_upsampled]).sample(
        frac=1, random_state=seed
    ).reset_index(drop=True)

df_train_balanced = oversample_minority(df_train, LABEL)
print(f"\nBalanced train: {len(df_train_balanced):,} rows | "
      f"SNAP distribution: {dict(Counter(df_train_balanced[LABEL]))}")

### Prompt Builders

In [ ]:
def build_structured_prompt(row, features, instruction=INSTRUCTION_STRUCTURED):
    user_content = "\n".join(
        f"{COL_MAP[k]}: {row[k]}" for k in features if k in row
    )
    return {
        "prompt":     [{"role": "system",    "content": instruction},
                       {"role": "user",      "content": user_content}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }


def build_narrative_prompt(row, features, instruction=INSTRUCTION_NARRATIVE, current_wave):
    age        = row.get("age", "unknown age")
    sex        = row.get("sex", "").lower()
    hispanic   = row.get("hispanic.origin", "")
    race       = row.get("race", "")
    ethnicity  = f"{hispanic} {race}".strip()
    marital    = row.get("marital.status", "").lower()
    edu        = row.get("education", "").lower()
    employment = row.get("employment", "").lower()
    state      = row.get("state", "")
    income     = row.get("income", "")
    citizenship = row.get("citizenship", "").lower()
    snap_w1    = row.get("SNAP_wave1", "Unknown")

    if sex == "female":
        person = "woman"
        pronoun = "She"
    elif sex == "male":
        person = "man"
        pronoun = "He"
    else:
        person = "person"
        pronoun = "They"

    ethnicity_parts = []
    if hispanic:
        ethnicity_parts.append(hispanic)
    if race:
        ethnicity_parts.append(race)
    ethnicity = ", ".join(ethnicity_parts)

    first_sentence_parts = [f"This is a {age}-year-old"]
    if ethnicity:
        first_sentence_parts.append(ethnicity)
    first_sentence_parts.append(person)
    first_sentence = " ".join(first_sentence_parts)

    if citizenship:
        first_sentence += f" who is a {citizenship}."
    else:
        first_sentence += "."

    second_sentence = (
        f"{pronoun} {'has' if pronoun == 'She' or pronoun == 'He' else 'have'} "
        f"{marital} and {'holds' if pronoun == 'She' or pronoun == 'He' else 'hold'} "
        f"{'an' if str(edu).lower()[0] in 'aeiou' else 'a'} {edu}."
        if marital and edu else ""
    )

    third_sentence = (
        f"During the current survey month, {pronoun.lower()} was {employment}, "
        f"with a monthly household income of {income}, and resides in {state}."
        if employment and income and state else ""
    )

    snap_sentences = []
    wave_labels = {
        1: "In the first survey wave",
        2: "In the second survey wave",
        3: "In the third survey wave",
        4: "In the fourth survey wave"
    }

    for w in range(1, current_wave):
        snap_value = str(row.get(f"SNAP_wave{w}", "Unknown")).strip().title()
        if snap_value == "Yes":
            snap_sentences.append(f"{wave_labels[w]}, {pronoun.lower()} was covered by SNAP.")
        elif snap_value == "No":
            snap_sentences.append(f"{wave_labels[w]}, {pronoun.lower()} was not covered by SNAP.")

    narrative = " ".join(
        s for s in [first_sentence, second_sentence, third_sentence] if s
    )

    if snap_sentences:
        narrative += " " + " ".join(snap_sentences)
    return {
        "prompt":     [{"role": "system",    "content": instruction},
                       {"role": "user",      "content": narrative}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }
    return {
        "prompt":     [{"role": "system",    "content": instruction},
                       {"role": "user",      "content": narrative}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }


def make_dataset(df, prompt_fn, features):
    from datasets import Dataset
    records = [prompt_fn(row, features) for _, row in df.iterrows()]
    return Dataset.from_list(records)

### Export to JSONL (MLX Format)

In [ ]:
def export_to_jsonl(df, prompt_fn, features, path):
    with open(path, "w") as f:
        for _, row in df.iterrows():
            sample   = prompt_fn(row, features)
            messages = sample["prompt"] + sample["completion"]
            f.write(json.dumps({"messages": messages}) + "\n")


# Experiment matrix — same as Wave 1, 2 and 3 structure T2A uses DEMO_FEATURES
# T2C_longitudinal_model uses SNAP_wave1, SNAP_wave2, SNAP_wave3 only as features.
experiments_data = {
    "T1A_imbalanced":   (df_train,          build_structured_prompt, FULL_FEATURES,     FULL_FEATURES),
    "T1B_oversampled":  (df_train_balanced,  build_structured_prompt, FULL_FEATURES,     FULL_FEATURES),
    "T2A_demo_only":    (df_train,          build_structured_prompt, DEMO_FEATURES,     DEMO_FEATURES),
    "T2C_longitudinal_model":   (df_train,          build_structured_prompt, LONGITUDINAL_FEATURES,     LONGITUDINAL_FEATURES),
    "T3A_narrative":    (df_train,          build_narrative_prompt,  FULL_FEATURES,     FULL_FEATURES),
}

os.makedirs("./mlx_data", exist_ok=True)
for run_label, (train_df, prompt_fn, train_feats, val_feats) in experiments_data.items():
    folder = f"./mlx_data/{run_label}"
    os.makedirs(folder, exist_ok=True)
    export_to_jsonl(train_df, prompt_fn, train_feats, f"{folder}/train.jsonl")
    export_to_jsonl(df_test,  prompt_fn, val_feats,   f"{folder}/valid.jsonl")
    print(f"Exported {folder}/")

### Making Test Datasets

In [ ]:
test_ds_structured_full     = make_dataset(df_test, build_structured_prompt, FULL_FEATURES)
test_ds_structured_demo     = make_dataset(df_test, build_structured_prompt, DEMO_FEATURES)
test_ds_structured_longitudinal  = make_dataset(df_test, build_structured_prompt, LONGITUDINAL_FEATURES)
test_ds_narrative_full      = make_dataset(df_test, build_narrative_prompt,  FULL_FEATURES)
print("Test datasets ready.")

### Defining Predictions

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report
)

def evaluate_predictions(predictions, references, label_name=""):
    valid         = [(p, r) for p, r in zip(predictions, references) if p in ["Yes", "No"]]
    unknown_count = len(predictions) - len(valid)
    if not valid:
        print("No valid predictions.")
        return {}
    y_pred, y_true = zip(*valid)
    metrics = {
        "Label":     label_name,
        "N_test":    len(df_test),
        "Accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "Recall":    round(recall_score(y_true, y_pred, pos_label="Yes"), 4),
        "Precision": round(precision_score(y_true, y_pred, pos_label="Yes", zero_division=0), 4),
        "F1":        round(f1_score(y_true, y_pred, pos_label="Yes"), 4),
        "Unknown":   unknown_count,
    }
    print(f"\n── {label_name} ──")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return metrics

### Calibration Analytics

In [ ]:
def calibration_analysis(probabilities, references, label_name):
    y_true_bin = [1 if r == "Yes" else 0 for r in references]
    probs      = np.array(probabilities)

    n_bins    = 10 # this can be changed to however many bins one wants
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece       = 0.0
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = np.mean(np.array(y_true_bin)[mask])
        bin_conf = np.mean(probs[mask])
        ece     += mask.sum() / len(probs) * abs(bin_acc - bin_conf)
        return round(ece, 4)

### Run Inference Function

In [ ]:
from mlx_lm import load, generate

def format_chat_prompt(prompt_list):
    return "".join(
        f"<|{m['role']}|>\n{m['content']}\n" for m in prompt_list
    ) + "<|assistant|>\n"


def run_inference_mlx(model, tokenizer, dataset, return_probs=False):
    predictions, references, probabilities = [], [], []

    for example in tqdm(dataset, desc="Generating"):
        prompt_text = format_chat_prompt(example["prompt"])
        response    = generate(
            model, tokenizer,
            prompt=prompt_text,
            max_tokens=5,
            verbose=False

        )
        match = re.search(r"\b(Yes|No)\b", response, re.IGNORECASE)
        predictions.append(match.group(1).title() if match else "Unknown")
        references.append(example["completion"][0]["content"].strip().title())

        if return_probs:
            if match:
                probabilities.append(1.0 if match.group(1).title() == "Yes" else 0.0)
            else:
                probabilities.append(0.5)

    if return_probs:
        return predictions, references, probabilities
    return predictions, references

### LoRA Configuration

In [ ]:
import yaml

lora_config = {
    "lora_parameters": {
        "rank":    16,
        "scale":   2.0,   # equivalent to alpha=32 with rank=16 (scale = alpha/rank)
        "dropout": 0.05,
    }
}

os.makedirs("./configs", exist_ok=True)
with open("./configs/lora_config.yaml", "w") as f:
    yaml.dump(lora_config, f)

print("LoRA config saved to ./configs/lora_config.yaml")

### Training and Inference for T1A_imbalanced

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T1A_imbalanced \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w4/T1A_imbalanced \
  --save-every 500 \
  -c ./configs/lora_config.yaml

In [ ]:
import time
#  Initialise accumulators on first experiment
all_metrics      = []
experiment_times = {}

#  Load model with T1A_imbalanced adapter
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w4/T1A_imbalanced",
)

# Inference
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

#  Evaluate
metrics = evaluate_predictions(preds, refs, label_name="T1A_imbalanced")

#  Calibration
ece = calibration_analysis(
    probs, refs,
    label_name="T1A_imbalanced"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

#  Save predictions
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T1A_imbalanced.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T1A_imbalanced"] = elapsed
print(f"\n  T1A_imbalanced done in {elapsed:.1f}s")

# Clean up
del model_mlx, tok_mlx

### Training and Inference for T1B_oversampled

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T1B_oversampled \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w4/T1B_oversampled \
    --save-every 500 \
    -c ./configs/lora_config.yaml

In [ ]:
#  Load model with T1B_oversampled adapter
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w4/T1B_oversampled",
)

#  Inference
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

#  Evaluate
metrics = evaluate_predictions(preds, refs, label_name="T1B_oversampled")

#  Calibration
ece = calibration_analysis(
    probs, refs,
    label_name="T1B_oversampled"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

#  Save predictions
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T1B_oversampled.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T1B_oversampled"] = elapsed
print(f"\n  T1B_oversampled done in {elapsed:.1f}s")

#  Clean up
del model_mlx, tok_mlx

### Training and Inference for T2A_demo_only

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T2A_demo_only \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w4/T2A_demo_only \
    --save-every 500 \
    -c ./configs/lora_config.yaml

In [ ]:
# Load model with T2A_demo_only adapter
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w4/T2A_demo_only",
)

#  Inference
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_demo, return_probs=True
)

# Evaluate
metrics = evaluate_predictions(preds, refs, label_name="T2A_demo_only")

# Calibration
ece = calibration_analysis(
    probs, refs,
    label_name="T2A_demo_only"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

#  Save predictions
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T2A_demo_only.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T2A_demo_only"] = elapsed
print(f"\n  T2A_demo_only done in {elapsed:.1f}s")

#  Clean up
del model_mlx, tok_mlx

### Training and Inference for T2C_longitudinal_model

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T2C_longitudinal_model \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w4/T2C_longitudinal_model \
    --save-every 500 \
    -c ./configs/lora_config.yaml

In [ ]:
# Load model with T2C_longitudinal_model adapter
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w4/T2C_longitudinal_model",
)

# Inference
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_longitudinal, return_probs=True
)

#  Evaluate
metrics = evaluate_predictions(preds, refs, label_name="T2C_longitudinal_model")

#  Calibration
ece = calibration_analysis(
    probs, refs,
    label_name="T2C_longitudinal_model"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

# Save predictions
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T2C_longitudinal_model.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T2C_longitudinal_model"] = elapsed
print(f"\n T2C_longitudinal_model done in {elapsed:.1f}s")

#  Clean up
del model_mlx, tok_mlx

### Inference for zero shot

In [ ]:
## Inference for zero shot
t0 = time.time()

model_mlx, tok_mlx = load(
    MODEL_ID,
    # no adapter_path — base model only
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T3A_zeroshot")

ece = calibration_analysis(
    probs, refs,
    label_name="T3A_zeroshot"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3A_zeroshot.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3A_zeroshot"] = elapsed
print(f"\n  T3A_zeroshot done in {elapsed:.1f}s")

del model_mlx, tok_mlx

### Training and Inference for T3A_narrative

In [ ]:
!mlx_lm.lora \
    --model {MODEL_ID} \
    --train \
    --data ./mlx_data/T3A_narrative \
    --num-layers 16 \
    --batch-size 4 \
    --grad-accumulation-steps 4 \
    --iters 2000 \
    --learning-rate 2e-4 \
    --mask-prompt \
    --adapter-path ./adapters_w4/T3A_narrative \
    --save-every 500 \
    -c ./configs/lora_config.yaml

In [ ]:
# Load model with T3A_narrative adapter
model_mlx, tok_mlx = load(
    MODEL_ID,
    adapter_path="./adapters_w4/T3A_narrative",
)

# Inference
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_narrative_full, return_probs=True
)

# Evaluate
metrics = evaluate_predictions(preds, refs, label_name="T3A_narrative")

#  Calibration
ece = calibration_analysis(
    probs, refs,
    label_name="T3A_narrative"
)
metrics["ECE"] = ece
all_metrics.append(metrics)

#  Save predictions
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3A_narrative.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3A_narrative"] = elapsed
print(f"\n  T3A_narrative done in {elapsed:.1f}s")

# Clean up
del model_mlx, tok_mlx

### Save Results Summary

In [ ]:
results_df = pd.DataFrame(all_metrics)
results_csv = os.path.join(OUTPUT_DIR, "results_wave4_summary.csv")
results_df.to_csv(results_csv, index=False)
print("\n── Final Results (Wave 4) ──")
print(results_df.to_string(index=False))
print(f"\nSaved → {results_csv}")

print("\n── Experiment Runtimes ──")
for exp, t in experiment_times.items():
    print(f"  {exp}: {t:.1f}s")